In [0]:
# DataSink: Base class for different data loading strategies
# Uses Factory pattern to create appropriate sink based on destination type
class DataSink:
  
    def __init__(self, df, path, method, para=None):
        # df: DataFrame to be saved
        # path: Destination path or table name
        # method: Write mode (e.g., 'overwrite', 'append')
        # para: Optional parameters dict for additional configurations
        self.df = df
        self.path = path
        self.method = method
        self.para = para
    
    def load_data_frame(self):
        # Abstract method - must be implemented by subclasses
        raise NotImplementedError("Not implemented")


# Concrete implementation for loading data to DBFS without partitioning
class LoadToDBFS(DataSink):

    def load_data_frame(self):
        # Simple write operation to DBFS path
        self.df.write.mode(self.method).save(self.path)


# Concrete implementation for loading data to DBFS with partitioning
class LoadToDBFSWithPartition(DataSink):
    
    def load_data_frame(self):
        # Extract partition column names from parameters
        partitionByColumns = self.para.get('partitionByColumns')
        # Write data partitioned by specified columns
        self.df.write \
            .mode(self.method) \
            .partitionBy(*partitionByColumns) \
            .save(self.path)

# Concrete implementation for loading data to Delta Lake table
class LoadToDeltaTable(DataSink):

    def load_data_frame(self):
        # Write DataFrame as Delta table format
        self.df.write \
            .format('delta')\
            .mode(self.method) \
            .saveAsTable(self.path)

# Factory function: Returns appropriate DataSink instance based on sink_type
def get_sink_source(sink_type, df, path, method, para=None):
    # sink_type: Type of destination ('dbfs', 'dbfs_wpartition', 'delta')
    # Returns: Appropriate DataSink subclass instance

    if sink_type == 'dbfs':
        # Simple DBFS write without partitioning
        return LoadToDBFS(df, path, method, para)

    elif sink_type == 'dbfs_wpartition':
        # DBFS write with partitioning support
        return LoadToDBFSWithPartition(df, path, method, para)
    elif sink_type == 'delta':
        # Delta Table write with partitioning support
        return LoadToDeltaTable(df, path, method, para)

    else:
        # Invalid sink type provided
        raise Exception("Invalid sink type")



